In [33]:
#
# Cell 1: load libraries
#

import numpy as np
from pathlib import Path
import pickle

#directory_path = Path("D:/ECG/X_augmented_1000") # chose the right path beforhand
#directory_path = Path("D:/ECG/X_augmented_800") # chose the right path beforhand
#directory_path = Path("D:/ECG/X_augmented_600") # chose the right path beforhand
#directory_path = Path("D:/ECG/X_augmented_400") # chose the right path beforhand
#directory_path = Path("D:/ECG/X_augmented_200") # chose the right path beforhand
directory_path = Path("D:/ECG/XY_diff_class_4") # chose the right path beforhand




In [35]:
#
# Cell 2: load X-data
#


#loaded_arrays = np.load(directory_path/'X_np_arrays.npz',allow_pickle=True)
#loaded_arrays = np.load(directory_path/'X_np_arrays_2.npz',allow_pickle=True)
loaded_arrays = np.load(directory_path/'X_np_arrays_3.npz',allow_pickle=True)

# chose the file to restore:
X_train_in = loaded_arrays['a']
X_val_in = loaded_arrays['b']
X_test_in = loaded_arrays['c']

print("X_train_in shape:", X_train_in.shape)
print("X_val_in:", X_val_in.shape)
print("X_test_in:", X_test_in.shape)

X_train_in shape: (14378, 1000, 12)
X_val_in: (4793, 1000, 12)
X_test_in: (2126, 1000, 12)


In [37]:
#
# Cell 3: modify X-data: chose sequence length and lead 
# Note: if number of leads >1 is chosen the models has to be adapted accordingly
#

#chose lead:
lead = 1
#chose lenght of sequence (1000, 800, 600, 400, 200)
seq_length = 1000

X_train = X_train_in[:, 0:seq_length,lead].copy()
X_val = X_val_in[:, 0:seq_length,lead].copy()
X_test = X_test_in[:,0:seq_length,lead].copy()

print("X_train shape:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train shape: (14378, 1000)
X_val: (4793, 1000)
X_test: (2126, 1000)


In [39]:
#
# Cell 4: load y-data 
#

#with open(directory_path/'Y_data.pkl', 'rb') as f:
#   Y_final_dfs = pickle.load(f) # deserialize using load()
#f.close()  

#with open(directory_path/'Y_data_2.pkl', 'rb') as f:
#   Y_final_dfs = pickle.load(f) # deserialize using load()
#f.close()

with open(directory_path/'Y_data_3.pkl', 'rb') as f:
   Y_final_dfs = pickle.load(f) # deserialize using load()
f.close()

#Y_final_dfs

In [41]:
#
# Cell 4a: for control purpose 
#

print("y_train shape:", len(Y_final_dfs['train_df']))
print("y_val:", len(Y_final_dfs['val_df']))
print("y_test:", len(Y_final_dfs['test_df']))

y_train shape: 14378
y_val: 4793
y_test: 2126


In [43]:
#
# Cell 4b: for control purpose
#


df_control=Y_final_dfs['val_df']
df_control[50:60]

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
4389,4409,7352.0,3,"[NST_, AFIB, STD_]","AFIB, NST_, STD_",3,AFIB_all_other
9318,9342,15328.0,3,"[ABQRS, SR]",ABQRS,1,ABQRS
13054,13086,20102.0,4,"[ABQRS, SR]",ABQRS,1,ABQRS
13548,13580,14982.0,9,"[ABQRS, SR]",ABQRS,1,ABQRS
17995,18033,5647.0,4,"[NST_, SR]",NST_,1,all_other_allone
7268,7289,21144.0,3,"[ABQRS, AFIB]","ABQRS, AFIB",2,ABQRS_AFIB
3666,3681,17066.0,2,[AFIB],AFIB,1,AFIB
15937,15975,7782.0,2,"[SBRAD, STD_, NT_]","NT_, SBRAD, STD_",3,all_other_allone
17465,17503,11348.0,3,"[ABQRS, SR]",ABQRS,1,ABQRS
4176,4196,20956.0,9,"[ABQRS, SR]",ABQRS,1,ABQRS


In [47]:
#
# Cell 5: converts the content of df, column 'codes_2', into a list 
#


#y_train=list(Y_final_dfs['train_df']['codes'])
#y_val=list(Y_final_dfs['val_df']['codes'])
#y_test=list(Y_final_dfs['test_df']['codes'])

y_train=list(Y_final_dfs['train_df']['codes_2'])
y_val=list(Y_final_dfs['val_df']['codes_2'])
y_test=list(Y_final_dfs['test_df']['codes_2'])

print("len y_train: ", len(y_train))
print("len y_val: ", len(y_val))
print("len y_test: ", len(y_test))

y_val [0:10]

len y_train:  14378
len y_val:  4793
len y_test:  2126


['SR',
 'AFIB',
 'SR',
 'all_other_allone',
 'all_other_allone',
 'SR',
 'SR',
 'all_other_allone',
 'SR',
 'all_other_allone']

In [49]:
#
# Cell 6: converts string labels into multi-hot labels 
#



y_string_dict = {
    'y_train': y_train,
    'y_val': y_val,
    'y_test': y_test
}

encoding = {
    'SR': [1, 0, 0, 0],
    'ABQRS': [0, 1, 0, 0],
    'AFIB': [0, 0, 1, 0],
    'all_other_allone': [0, 0, 0, 1],
    'ABQRS_all_other': [0, 1, 0, 1],
    'AFIB_all_other': [0, 0, 1, 1],
    'ABQRS_AFIB': [0, 1, 1, 0],
    'intersection_all': [0, 1, 1, 1]
}

y_multihot_dict = {
    key: np.array(
        [encoding[label] for label in labels],
        dtype=np.float32
    )
    for key, labels in y_string_dict.items()
}
y_multihot_dict['y_train'][0:10]

array([[1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       [0., 1., 0., 1.],
       [1., 0., 0., 0.]], dtype=float32)

In [51]:
#
# Cell 7: calculates positive weight of labels 
#

y_train_ml = y_multihot_dict["y_train"]

label_names = ["SR", "ABQRS", "AFIB", "all_other"]

n_pos = np.sum(y_train_ml, axis=0)
n_neg = len(y_train_ml) - n_pos

pos_weight = n_neg / n_pos


for label, pos, neg, weight in zip(
    label_names, n_pos, n_neg, pos_weight
):
    print(
        f"{label:10s}: "
        f"positive={int(pos):5d}, "
        f"negative={int(neg):5d}, "
        f"pos_weight={weight:.3f}"
    )

SR        : positive= 6834, negative= 7544, pos_weight=1.104
ABQRS     : positive= 2254, negative=12124, pos_weight=5.379
AFIB      : positive= 1021, negative=13357, pos_weight=13.082
all_other : positive= 5495, negative= 8883, pos_weight=1.617


In [53]:
#
# Cell 8: release the y-values from dictionary 
#

y_train = y_multihot_dict["y_train"]
y_val = y_multihot_dict["y_val"]
y_test = y_multihot_dict["y_test"]

In [55]:
#
# Cell 8a: for control purpose 
#

print("len y_train: ", len(y_train))
print("len y_val: ", len(y_val))
print("len y_test: ", len(y_test))

len y_train:  14378
len y_val:  4793
len y_test:  2126


In [57]:
#
# Cell 9: final preparation of the input data for model training 
#

# -----------------------------
# 0) Prepare data shapes
# -----------------------------

def ensure_3d(X):
    X = np.asarray(X)
    if X.ndim == 2:          # (N, 1000) -> (N, 1000, 1)
        X = X[..., None]
    return X

# Use your arrays here:
# X_train, y_train (0/1), X_val, y_val (0/1)
X_train = ensure_3d(X_train).astype(np.float32)
X_val   = ensure_3d(X_val).astype(np.float32)
X_test   = ensure_3d(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.int32)
y_val   = np.asarray(y_val).astype(np.int32)
y_test  = np.asarray(y_test).astype(np.int32)

input_shape = X_train.shape[1:]  # (1000, 1)



In [59]:
#
# Cell 9a: for control purpose 
#


print("X_train shape:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

X_train shape: (14378, 1000, 1)
X_val: (4793, 1000, 1)
X_test: (2126, 1000, 1)
y_train shape: (14378, 4)
y_val shape: (4793, 4)
y_test shape: (2126, 4)


In [61]:
#
# Cell 10: creates a folder for data storage.
#

directory_path_input = Path("D:/ECG/input_for_model") # chose the right path beforhand

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path_input}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path_input}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_pathinput}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\input_for_model' already exists.


In [63]:
#
# Cell 11: store data.
#

np.savez_compressed(directory_path_input/'X_y_input.npz', a=X_train, b=y_train, c=X_val, d=y_val, e=X_test, f=y_test)
